# Step 11 - Rewrite recommender, LLM step for appearance-type implicit bias

**Status: pilot of a fixed pipeline. No human evaluation here (assigned to the research partner). The automatic checks use our own rules and the Step 9 model, so they show whether a rewrite passes our guardrails, not whether it reads well or is neutral to a person.**

Pipeline (fixed order, no agent, no retries):
1. **Select**: units that Step 9 labelled `implicit` and that match the Step 6 appearance pattern (looks, "pleasing personality", grooming, physical fitness, height, weight).
2. **Route**: height or weight conditions are never sent to the model (`manual_review_measurement`); a person decides if they are a genuine job requirement.
3. **Rewrite**: one prompted open-weights model rewrites the single line with a fixed few-shot prompt and greedy decoding.
4. **Check**: rule-based guardrails accept or reject the output (section 4). Rejected outputs are kept and reported, never silently replaced.
5. Every other line of an ad is left as written.

Scope: appearance-type implicit lines only. Gender-coded duty words (leadership, dependable, honest, ...) are **not** rewritten, because they may describe the job and no human label separates them yet. Explicit bias is handled by the Step 10 rules.

Inputs: `outputs/equal_step9_silver_baseline/step9_silver_labels.csv`, `step9_silver_model.joblib`. Outputs: `outputs/equal_step11_llm_rewrite/`.

## 1. Environment, model and decoding settings (reproducibility log)

In [1]:
import difflib, json, os, platform, re, sys, time
from pathlib import Path

import joblib, numpy as np, pandas as pd, sklearn, torch, transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

OUT = Path("outputs") / "equal_step11_llm_rewrite"
OUT.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
MODEL_LICENSE = "apache-2.0 (from the Hugging Face model card metadata)"
MAX_NEW_TOKENS = 40
torch.manual_seed(42)

tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_REVISION, local_files_only=True, dtype=torch.float32).eval()
n_params = sum(p.numel() for p in model.parameters())

GEN = dict(do_sample=False, temperature=None, top_p=None, top_k=None, repetition_penalty=1.0, max_new_tokens=MAX_NEW_TOKENS)
env = {"python": sys.version.split()[0], "platform": platform.platform(), "processor": platform.processor(),
       "logical_cpus": os.cpu_count(), "torch_threads": torch.get_num_threads(), "gpu_used": False,
       "numpy": np.__version__, "pandas": pd.__version__, "scikit_learn": sklearn.__version__,
       "torch": torch.__version__, "transformers": transformers.__version__,
       "model_id": MODEL_ID, "model_revision": MODEL_REVISION, "model_license": MODEL_LICENSE,
       "model_parameters": int(n_params), "precision": "float32",
       "decoding": "greedy (do_sample=False), repetition_penalty=1.0, max_new_tokens=%d" % MAX_NEW_TOKENS,
       "seed": 42, "retries": "none"}
try:
    import psutil
    env["ram_gb"] = round(psutil.virtual_memory().total / 2**30, 1)
except ImportError:
    env["ram_gb"] = "not captured (psutil not installed)"
(OUT / "step11_environment.json").write_text(json.dumps(env, indent=2), encoding="utf-8")
print(json.dumps(env, indent=2))

d:\Dev\project-equal\equal-v2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 338/338 [00:03<00:00, 93.83it/s] 

{
  "python": "3.12.9",
  "platform": "Windows-11-10.0.26200-SP0",
  "processor": "AMD64 Family 23 Model 113 Stepping 0, AuthenticAMD",
  "logical_cpus": 12,
  "torch_threads": 6,
  "gpu_used": false,
  "numpy": "2.5.4",
  "pandas": "3.0.6",
  "scikit_learn": "1.9.1",
  "torch": "2.14.1+cpu",
  "transformers": "5.20.2",
  "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
  "model_revision": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
  "model_license": "apache-2.0 (from the Hugging Face model card metadata)",
  "model_parameters": 1543714304,
  "precision": "float32",
  "decoding": "greedy (do_sample=False), repetition_penalty=1.0, max_new_tokens=40",
  "seed": 42,
  "retries": "none",
  "ram_gb": 31.9
}


## 2. Select the units and route them

Regexes and coded-word stems are copied from Steps 6 and 9. A unit is **in scope** if the Step 9 silver label is `implicit` and the appearance pattern matches.

In [2]:
MASC = ("active adventurous aggress ambitio analy assert athlet autonom boast challeng compet confident "
        "courag decisive decision determin dominan driven fearless force headstrong hierarch impulsive "
        "independen individual intellect lead logic objective opinion outspoken persist self-relian "
        "self-sufficien stubborn superior unreasonab").split()
FEM = ("agree affectionate child cheer collab commit communal compassion connect considerate cooperat "
       "depend emotiona empath feel flatterable gentle honest interpersonal interdependen kind kinship "
       "loyal modest nurtur pleasant polite quiet respons sensitiv submissive support sympath tender "
       "together trust understand warm enthusias inclusive yield share").split()
FALSE_POSITIVE_PREFIXES = ("confidential", "committee", "shareholder", "kinds")

def coded_words(text, stems):
    hits = []
    for w in re.findall(r"[a-z][a-z-]*", text.lower()):
        if w.startswith(FALSE_POSITIVE_PREFIXES):
            continue
        parts = [w] + (w.split("-") if "-" in w else [])
        if any(p.startswith(s) for p in parts for s in stems):
            hits.append(w)
    return hits

MALE_RE = re.compile(r"\b(?:male|males|men|gentlemen)\b", re.I)
FEMALE_RE = re.compile(r"\b(?:female|females|women|ladies)\b", re.I)
MARITAL_RE = re.compile(r"\b(?:single|married|unmarried|widow(?:ed|er)?|marital|pregnan\w*)\b", re.I)
AGE_RE = re.compile(r"\b(?:\d{2}\s*(?:-|to)\s*\d{2}\s*(?:years|yrs|yo|y/o|years\s+old)|"
                    r"(?:below|under|not\s+(?:more|older)\s+than|up\s+to|maximum\s+(?:age\s+)?of)\s*\d{2}|"
                    r"age[sd]?\s*(?:range|limit|requirement)?\s*[:\-]?\s*\d{2})\b", re.I)
APPEAR_RE = re.compile(r"\b(?:good\s+looking|pleasing\s+personality|presentable|attractive|"
                       r"height|weight|physically\s+fit|heavy\s+lifting|tall|slim)\b", re.I)
MEASURE_RE = re.compile(r"\b(?:height|heights|weigh|weight|centimeters?|kilograms?|kgs?|cm)\b|\d\s*['’′]\s*\d|\d\s*['’′]\s*(?:\d|height|tall)", re.I)
APPEAR_PHRASE_RE = re.compile(r"\b(?:good\s+looking|pleasing\s+personality|presentable(?:\s+looks)?|attractive|physically\s+fit|fair\s+complexion|well\s+groomed|slim|tall)\b", re.I)

lab = pd.read_csv("outputs/equal_step9_silver_baseline/step9_silver_labels.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
imp = lab[lab["silver_bias_type"] == "implicit"]
scope = imp[imp["sentence"].map(lambda s: bool(APPEAR_RE.search(s)))].copy()
scope["route"] = np.where(scope["sentence"].map(lambda s: bool(MEASURE_RE.search(s))), "manual_review_measurement", "llm")
print("silver implicit units:", len(imp), "| appearance-type (in scope):", len(scope), "| distinct texts:", scope["sentence"].nunique())
print("not rewritten (gender-coded duty words and other implicit):", len(imp) - len(scope))
display(scope.groupby("route").agg(units=("sentence_id", "size"), distinct_texts=("sentence", "nunique")))

silver implicit units: 668 | appearance-type (in scope): 91 | distinct texts: 37
not rewritten (gender-coded duty words and other implicit): 577


,units,distinct_texts
route,,
llm,78,28
manual_review_measurement,13,9


## 3. Prompt and generation

The system prompt and the five few-shot examples are **written by us** and are not taken from the corpus, so no evaluated unit is in the prompt. The model sees only the single line, never the rest of the ad. Output `DELETE` means the line held only appearance wording.

In [3]:
SYSTEM = ("You edit one line of a job advertisement. The line contains wording about a person's appearance or looks "
          "(for example looks, a 'pleasing' personality, grooming, physical fitness) that is not a stated job requirement. "
          "Rewrite the line so that it contains no appearance wording. Keep every other requirement in the line exactly as written. "
          "Do not add new requirements, numbers or details. If the line contains only appearance wording and nothing else, answer DELETE. "
          "If the appearance wording describes a physical task the job needs (for example lifting or walking), restate it as the ability "
          "to do that task. Answer with the rewritten line only.")
FEWSHOT = [("Good looking and with excellent typing skills", "With excellent typing skills"),
           ("Attractive", "DELETE"),
           ("Slim and able to work under pressure", "Able to work under pressure"),
           ("Must be physically fit to carry boxes", "Must be able to carry boxes"),
           ("Well groomed and has a driver's license", "Has a driver's license")]
PROMPT_VERSION = "v1"

def llm_rewrite(unit):
    msgs = [{"role": "system", "content": SYSTEM}]
    for a, b in FEWSHOT:
        msgs += [{"role": "user", "content": a}, {"role": "assistant", "content": b}]
    msgs.append({"role": "user", "content": unit})
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = model.generate(**enc, **GEN)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

(OUT / "step11_prompt.json").write_text(json.dumps({"version": PROMPT_VERSION, "system": SYSTEM, "few_shot": FEWSHOT}, indent=2, ensure_ascii=False), encoding="utf-8")
t0 = time.time(); demo = llm_rewrite("With pleasing personality"); print(repr(demo), f"({time.time() - t0:.1f}s)")

'PLEASANT PERSONALITY' (6.3s)


## 4. Guardrail checks

A rewrite is **accepted by the checks** only if all hard checks pass. The thresholds below are design choices made by us, not validated values.

| Check | Hard rule |
|---|---|
| `empty` | output must not be empty |
| `multi_line` | output must be one line |
| `delete_loses_content` | `DELETE` is allowed only if nothing but appearance wording and filler remains |
| `appearance_remains` | the appearance pattern must not match the output |
| `protected_wording_added` | no gender, marital or age wording in the output |
| `new_coded_word` | no gender-coded stem that was not already in the input |
| `new_number` | no digits that were not in the input |
| `too_long` | at most 20 words and at most 6 words longer than the input |
| `content_dropped` | at least 70% of the other content words of the input must still appear (prefix match, first 5 letters) |

Not a hard rule: the Step 9 model label of the output and the edit similarity. They are reported only, because the Step 9 model is trained on rule labels and gave false positives on boilerplate.

In [4]:
STOP = set("a an the and or of to with have has had must be is are can able in on for at as by who that this their his her our applicant applicants candidate candidates should will may also well good".split())

def content_tokens(s):
    return [w for w in re.findall(r"[a-z]+", s.lower()) if w not in STOP and len(w) > 2]

def residual_tokens(orig):
    return content_tokens(APPEAR_PHRASE_RE.sub(" ", orig))

def check_rewrite(orig, raw):
    reasons = []
    out = raw.strip().strip('"').strip("'").strip()
    if "\n" in out:
        reasons.append("multi_line"); out = out.splitlines()[0].strip()
    if not out:
        return out, ["empty"], None
    is_delete = out.upper() == "DELETE"
    resid = residual_tokens(orig)
    if is_delete:
        if resid:
            reasons.append("delete_loses_content")
        return ("" if not reasons else out), reasons, 1.0 if not reasons else None
    if APPEAR_RE.search(out):
        reasons.append("appearance_remains")
    if MALE_RE.search(out) or FEMALE_RE.search(out) or MARITAL_RE.search(out) or AGE_RE.search(out):
        reasons.append("protected_wording_added")
    if set(coded_words(out, MASC + FEM)) - set(coded_words(orig, MASC + FEM)):
        reasons.append("new_coded_word")
    if set(re.findall(r"\d+", out)) - set(re.findall(r"\d+", orig)):
        reasons.append("new_number")
    n_out, n_in = len(out.split()), len(orig.split())
    if n_out > 20 or n_out > n_in + 6:
        reasons.append("too_long")
    if resid:
        out_pre = {w[:5] for w in content_tokens(out)}
        kept = sum(w[:5] in out_pre for w in resid) / len(resid)
        if kept < 0.7:
            reasons.append("content_dropped")
    return out, reasons, None

for o, r in [("Physically fit and have a good moral character", "Have a good moral character"),
             ("With pleasing personality", "With pleasant manner"),
             ("With pleasing personality", "DELETE"),
             ("Physically fit and have a good moral character", "DELETE")]:
    print(f"{o!r} -> {r!r}: {check_rewrite(o, r)[:2]}")

'Physically fit and have a good moral character' -> 'Have a good moral character': ('Have a good moral character', [])
'With pleasing personality' -> 'With pleasant manner': ('With pleasant manner', ['new_coded_word'])
'With pleasing personality' -> 'DELETE': ('', [])
'Physically fit and have a good moral character' -> 'DELETE': ('DELETE', ['delete_loses_content'])


## 5. Run on the distinct in-scope texts

In [5]:
distinct = scope[["sentence", "route"]].drop_duplicates("sentence").reset_index(drop=True)
clf_bundle = joblib.load("outputs/equal_step9_silver_baseline/step9_silver_model.joblib")
vec, clf = clf_bundle["vectorizer"], clf_bundle["classifier"]

rows = []
for _, r in distinct.iterrows():
    s = r["sentence"]
    if r["route"] == "manual_review_measurement":
        rows.append({"sentence": s, "route": r["route"], "raw_output": "", "rewrite": s, "status": "manual_review_measurement",
                     "reasons": "", "seconds": 0.0})
        continue
    t0 = time.time(); raw = llm_rewrite(s); dt = time.time() - t0
    out, reasons, _ = check_rewrite(s, raw)
    status = "accepted_by_checks" if not reasons else "rejected_by_checks"
    rows.append({"sentence": s, "route": r["route"], "raw_output": raw, "rewrite": out if not reasons else s, "status": status,
                 "reasons": ", ".join(reasons), "seconds": round(dt, 1), "candidate": out})
res = pd.DataFrame(rows)
cand = res["candidate"].fillna("") if "candidate" in res else pd.Series([""] * len(res))
mask = (res["status"] != "manual_review_measurement") & cand.ne("")
res["model_label_of_candidate"] = ""
res.loc[mask, "model_label_of_candidate"] = clf.predict(vec.transform(cand[mask]))
res["edit_similarity"] = [round(difflib.SequenceMatcher(None, a.lower(), b.lower()).ratio(), 3) if b else "" for a, b in zip(res["sentence"], cand)]
res.to_csv(OUT / "step11_rewrite_distinct_texts.csv", index=False, encoding="utf-8-sig")
print("distinct texts:", len(res), "| total generation seconds:", round(res["seconds"].sum(), 1), "| mean seconds per generation:", round(res.loc[res["route"] == "llm", "seconds"].mean(), 1))

distinct texts: 37 | total generation seconds: 150.3 | mean seconds per generation: 5.4


In [6]:
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 250)
cnt = res.merge(scope.groupby("sentence").size().rename("occurrences").reset_index(), on="sentence")
by_status = cnt.groupby("status").agg(distinct_texts=("sentence", "size"), units=("occurrences", "sum"))
display(by_status)
rej = cnt[cnt["status"] == "rejected_by_checks"]
print("rejection reasons (distinct texts):"); print(rej["reasons"].str.split(", ").explode().value_counts().to_string() if len(rej) else "none")
by_status.to_csv(OUT / "step11_checks_summary.csv")
display(cnt[["status", "sentence", "raw_output", "reasons", "occurrences"]].sort_values(["status", "occurrences"], ascending=[True, False]).reset_index(drop=True))

,distinct_texts,units
status,,
accepted_by_checks,5,17
manual_review_measurement,9,13
rejected_by_checks,23,61


rejection reasons (distinct texts):
reasons
appearance_remains    21
content_dropped        9
new_coded_word         1


,status,sentence,raw_output,reasons,occurrences
0,accepted_by_checks,Physically fit,DELETE,,10
1,accepted_by_checks,Must be physically fit,DELETE,,3
2,accepted_by_checks,Physically Fit,DELETE,,2
3,accepted_by_checks,Must be physically fit to walk long distances.,Must be able to walk long distances.,,1
4,accepted_by_checks,Physically fit.,DELETE,,1
5,manual_review_measurement,"At least 5'4"" in height",,,3
6,manual_review_measurement,"Height at least 5'1""",,,2
7,manual_review_measurement,At least 5’2” in height,,,2
8,manual_review_measurement,"At least 5'3"" in height",,,1
9,manual_review_measurement,Must weigh not more or less than five kilograms (5kgs) from the standard wei...,,,1


### 5b. Determinism check

The first 8 in-scope texts are generated a second time. With greedy decoding on the same machine the outputs should be identical.

In [7]:
first = res[res["route"] == "llm"].head(8)
again = [llm_rewrite(s) for s in first["sentence"]]
same = [a == b for a, b in zip(first["raw_output"], again)]
print("identical on rerun:", sum(same), "of", len(same))

identical on rerun: 8 of 8


## 6. Test: one ad and several ads

`rewrite_ads_llm` keeps every line and bullet exactly as written. Only lines that match the appearance pattern are processed: measurements go to manual review, other lines go through the model and the checks. A rejected output leaves the original line in place and logs the reason. Explicit lines (gender, marital, age) are only reported here; they are rewritten by the Step 10 rules. The ads below are written for the test and are not from the dataset, except the Puregold and Yamaha examples supplied by the team.

In [8]:
BULLET_RE = re.compile(r"^(\s*(?:[~\u2022\u25cf\u25aa\u25e6\u2023\u25cb\u25a0\u25a1\u25c6\u27a2\u27a4\u2713\u2714]+|[-*](?=\s|$))?\s*)(.*)$")
EXPLICIT_HINT_RE = re.compile("|".join(r.pattern for r in (MALE_RE, FEMALE_RE, MARITAL_RE, AGE_RE)), re.I)

def rewrite_ad_llm(text, ad_id="ad_1"):
    out, log = [], []
    for line in str(text).splitlines():
        m = BULLET_RE.match(line); prefix, body = m.group(1), m.group(2)
        if body and APPEAR_RE.search(body) and not EXPLICIT_HINT_RE.search(body):
            if MEASURE_RE.search(body):
                out.append(line); log.append({"ad_id": ad_id, "original": body, "raw_output": "", "rewrite": body, "status": "manual_review_measurement", "reasons": ""})
                continue
            raw = llm_rewrite(body); new, reasons, _ = check_rewrite(body, raw)
            if reasons:
                out.append(line); log.append({"ad_id": ad_id, "original": body, "raw_output": raw, "rewrite": body, "status": "rejected_by_checks", "reasons": ", ".join(reasons)})
            else:
                if new:
                    out.append(prefix + new)
                log.append({"ad_id": ad_id, "original": body, "raw_output": raw, "rewrite": new, "status": "accepted_by_checks", "reasons": ""})
        else:
            if body and EXPLICIT_HINT_RE.search(body):
                log.append({"ad_id": ad_id, "original": body, "raw_output": "", "rewrite": body, "status": "explicit_line_left_for_step10_rules", "reasons": ""})
            out.append(line)
    return "\n".join(out), pd.DataFrame(log, columns=["ad_id", "original", "raw_output", "rewrite", "status", "reasons"])

def rewrite_ads_llm(ads):
    if not isinstance(ads, dict):
        ads = {f"ad_{i + 1}": a for i, a in enumerate(ads)}
    texts, logs = {}, []
    for k, t in ads.items():
        texts[k], lg = rewrite_ad_llm(t, k); logs.append(lg)
    return texts, pd.concat(logs, ignore_index=True)

SINGLE = """Sales Associate

Qualifications:
~With pleasing personality
~At least 6 months of retail experience
~Physically fit and have a good moral character
~At least 5'4" in height

Interested applicants may send their resumes to:
~hr@example.com"""
txt, lg = rewrite_ad_llm(SINGLE, "single_test")
print(txt); print(); display(lg)
o, n = SINGLE.splitlines(), txt.splitlines()
print("lines before/after:", len(o), len(n), "| identical lines:", sum(a == b for a, b in zip(o, n)) if len(o) == len(n) else "line count changed")

Sales Associate

Qualifications:
~With pleasing personality
~At least 6 months of retail experience
~Physically fit and have a good moral character
~At least 5'4" in height

Interested applicants may send their resumes to:
~hr@example.com



,ad_id,original,raw_output,rewrite,status,reasons
0,single_test,With pleasing personality,PLEASANT PERSONALITY,With pleasing personality,rejected_by_checks,new_coded_word
1,single_test,Physically fit and have a good moral character,Physically fit,Physically fit and have a good moral character,rejected_by_checks,"appearance_remains, content_dropped"
2,single_test,"At least 5'4"" in height",,"At least 5'4"" in height",manual_review_measurement,


lines before/after: 10 10 | identical lines: 10


In [9]:
MULTI = {
    "single_test": SINGLE,
    "yamaha": "Yamaha\n\nProduction Associates (70)\n\nQualifications:\n~At least High School Graduate or 2-year Vocational Graduate\n~With or without work experience\n~Must be willing to render overtime",
    "retail_test": "Cashier\n\nQualifications:\n~Female, single\n~Well groomed, presentable and confident speaker\n~Applicants must have a pleasing personality and able to interact with people well\n~Must be physically fit to walk long distances",
}
texts, multi_log = rewrite_ads_llm(MULTI)
multi_log.to_csv(OUT / "step11_test_rewrite_log.csv", index=False, encoding="utf-8-sig")
display(multi_log)
print(texts["retail_test"])

,ad_id,original,raw_output,rewrite,status,reasons
0,single_test,With pleasing personality,PLEASANT PERSONALITY,With pleasing personality,rejected_by_checks,new_coded_word
1,single_test,Physically fit and have a good moral character,Physically fit,Physically fit and have a good moral character,rejected_by_checks,"appearance_remains, content_dropped"
2,single_test,"At least 5'4"" in height",,"At least 5'4"" in height",manual_review_measurement,
3,retail_test,"Female, single",,"Female, single",explicit_line_left_for_step10_rules,
4,retail_test,"Well groomed, presentable and confident speaker",Presentable and confident speaker,"Well groomed, presentable and confident speaker",rejected_by_checks,appearance_remains
5,retail_test,Applicants must have a pleasing personality and able to interact with people...,Applicants must have a pleasing personality,Applicants must have a pleasing personality and able to interact with people...,rejected_by_checks,"appearance_remains, content_dropped"
6,retail_test,Must be physically fit to walk long distances,Must be able to walk long distances,Must be able to walk long distances,accepted_by_checks,


Cashier

Qualifications:
~Female, single
~Well groomed, presentable and confident speaker
~Applicants must have a pleasing personality and able to interact with people well
~Must be able to walk long distances


## 7. Limitations

- One small model (1.5 billion parameters), one prompt, greedy decoding, no retries. Other models and prompts were not compared.
- The few-shot examples and the check thresholds are our own choices and are not validated.
- The checks use our rules and the Step 9 model, which was trained on those rules, so they cannot find bias the rules miss and say nothing about neutrality, readability or fluency. Human evaluation is carried out separately by the research partner.
- "Appearance wording" is defined by our own pattern. A physical requirement may be a genuine job requirement; the pipeline treats measurements as manual review and only restates "physically fit" as an ability.
- Only appearance-type implicit lines are processed. Gender-coded duty words (about 360 other silver-implicit units) and all explicit lines are out of scope here.
- Results are for the masked Step 2 units; employer and contact blocks are held separately.